# VGG16 Training on CIFAR-10 - Native FP8, Kernel Fusion, and Memory Padding

Notebook ini melatih VGG16 pada CIFAR-10 menggunakan optimalisasi tingkat rendah:
1. **FP8 Native Convolution & Linear**: Menggunakan custom layer `NativeConv2d` (im2col + `torch._scaled_mm`) and `NativeLinear` di dalam `ext3.nn.nn_native`.
2. **Adaptive Precision Layering**: Menggunakan `APAManager` untuk mendeteksi overflow secara real-time dan menurunkan presisi jika tidak stabil.
3. **Kernel Fusion (`torch.compile`)**: Mengompilasi model untuk memfusikan operasi amax, scale, cast (FP32 <-> FP8) langsung ke dalam kernel GPU Triton.
4. **Memory Alignment Padding**: Melakukan padding pada output logits classifier menjadi 16 kelas (agar Tensor Core FP8 aktif) dan memotongnya (*slicing*) kembali secara efisien ke 10 kelas sebelum masuk loss.

In [ ]:
import torch
import time
import os

# Menonaktifkan TF32 agar eksperimen presisi terkontrol
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Standarisasi Transform Data CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
import torch.nn as nn
from ext3.nn.nn_native import NativeConv2d, NativeLinear
from ext3.util.apa_manager import APAManager

# Arsitektur VGG16 yang dioptimalkan dengan FP8, Memory Alignment Padding & Slicing
class VGG16_CIFAR10_APA(nn.Module):
    def __init__(self, num_classes=10):
        super(VGG16_CIFAR10_APA, self).__init__()
        self.features = self._make_layers()
        self.classifier = nn.Sequential(
            NativeLinear(512, 4096, dtype_fwd=torch.float8_e4m3fn),
            nn.ReLU(True),
            nn.Dropout(),
            NativeLinear(4096, 4096, dtype_fwd=torch.float8_e4m3fn),
            nn.ReLU(True),
            nn.Dropout(),
            # Output dipad ke 16 (kelipatan 16) agar Tensor Core FP8 menyala
            NativeLinear(4096, 16, dtype_fwd=torch.float8_e4m3fn)
        )
        self.num_classes = num_classes

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        logits = self.classifier(out)
        
        # Melakukan slicing (view-only) secara efisien ke jumlah kelas asli (10)
        # Slicing ini adalah operasi zero-copy metadata-only di PyTorch
        return logits[:, :self.num_classes]

    def _make_layers(self):
        cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M']
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [
                    NativeConv2d(in_channels, x, kernel_size=3, padding=1, dtype_fwd=torch.float8_e4m3fn),
                    nn.BatchNorm2d(x), # Layer memory-bound & stabil tetap FP32
                    nn.ReLU(True)
                ]
                in_channels = x
        return nn.Sequential(*layers)

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
    accuracy = 100.0 * correct / total
    avg_loss = running_loss / total
    return accuracy, avg_loss

In [ ]:
# Inisialisasi Model standar
model = VGG16_CIFAR10_APA().to(device)

# KERNEL FUSION: Compile model menggunakan torch.compile() dengan mode max-autotune
# Ini menyatukan pencarian amax, casting, im2col, dan scaled_mm ke dalam CUDA kernel Triton tunggal.
compiled_model = torch.compile(model, mode="max-autotune")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(compiled_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

# Inisialisasi Adaptive Precision Manager
apa_manager = APAManager(model, initial_precision="fp8", overflow_threshold=3)
epochs = 5

In [ ]:
print("Memulai pelatihan dengan Native FP8 (Optimized & Compiled)...")
for epoch in range(1, epochs + 1):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        
    start_time = time.time()
    compiled_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = compiled_model(inputs)
        
        # Loss dihitung dari outputs yang sudah dipotong (slicing) ke 10 kelas
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        # APA Manager memantau stabilitas gradien & loss secara on-the-fly
        precision_changed = apa_manager.step(loss)
        if precision_changed:
            # Jika manajer memodifikasi model secara in-place, lakukan rekompilasi model
            compiled_model = torch.compile(model, mode="max-autotune")
            optimizer = torch.optim.SGD(compiled_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    epoch_time = time.time() - start_time
    train_acc = 100.0 * correct / total
    train_loss = running_loss / total
    
    # Evaluasi test accuracy
    test_acc, test_loss = evaluate(compiled_model, test_loader, criterion, device)
    
    # Profiling GPU VRAM
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
        print(f"Epoch [{epoch}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}% - "
              f"Peak VRAM: {peak_memory:.2f} MB")
    else:
        print(f"Epoch [{epoch}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}% - "
              f"Peak VRAM: N/A (CPU)")